# Vanishing Gradient : RNN Vanilla vs LSTM
## Le problème du gradient qui disparaît et la solution du *Constant Error Carousel*

---

Ce notebook explore mathématiquement et empiriquement pourquoi les **RNN vanilla** souffrent du vanishing gradient,  
et comment les **LSTM** résolvent ce problème grâce au mécanisme de *Constant Error Carousel* (CEC).

### Plan
1. Rappel mathématique : BPTT & vanishing gradient dans les RNN
2. Analyse du gradient dans un RNN vanilla (visualisation)
3. Architecture LSTM et le Constant Error Carousel
4. Analyse du gradient dans un LSTM
5. Expérience : tâche de mémorisation à long terme
6. Visualisation des normes de gradient couche par couche
7. Conclusion

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams.update({
    'figure.facecolor': '#0f0f1a',
    'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#444466',
    'axes.labelcolor': '#ccccee',
    'xtick.color': '#ccccee',
    'ytick.color': '#ccccee',
    'text.color': '#ccccee',
    'grid.color': '#333355',
    'grid.alpha': 0.5,
    'font.family': 'monospace',
    'font.size': 11,
})

COLORS = {
    'rnn':   '#ff6b6b',
    'lstm':  '#4ecdc4',
    'gold':  '#ffe66d',
    'purple':'#c77dff',
    'blue':  '#4895ef',
    'green': '#56cfe1',
}

torch.manual_seed(42)
np.random.seed(42)
print(' Librairies chargées')

---
## 1. Mathématiques : BPTT et Vanishing Gradient

### RNN Vanilla

Un RNN vanilla calcule :

$$h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$$

Lors du **Backpropagation Through Time (BPTT)**, le gradient de la loss $\mathcal{L}$ par rapport à $h_t$ se propage en arrière :

$$\frac{\partial \mathcal{L}}{\partial h_t} = \frac{\partial \mathcal{L}}{\partial h_T} \cdot \prod_{k=t}^{T-1} \frac{\partial h_{k+1}}{\partial h_k}$$

Chaque terme du produit vaut :

$$\frac{\partial h_{k+1}}{\partial h_k} = \text{diag}(1 - h_{k+1}^2) \cdot W_h$$

Le **problème** : si $\|W_h\| < 1$ ou que les dérivées de $\tanh$ sont petites, ce produit de $(T-t)$ matrices → **0 exponentiellement**.

### Condition de vanishing

La norme du gradient decroît exponentiellement si la **valeur singulière maximale** de la matrice Jacobienne est $< 1$ :

$$\left\|\frac{\partial h_{t+1}}{\partial h_t}\right\| \leq \underbrace{\|W_h\|}_{\text{dépend des poids}} \cdot \underbrace{\max_i (1 - h_i^2)}_{\leq 1 \text{ pour } \tanh}$$

In [ ]:
# ============================================================
# Visualisation théorique : produit de Jacobiennes
# ============================================================

def simulate_gradient_norm(hidden_size=64, T=100, n_trials=50, init_std=0.1):
    """Simule la norme du gradient à travers T pas de temps pour un RNN vanilla."""
    norms_all = []
    for _ in range(n_trials):
        # Initialisation de Wh (crucial : variance faible → vanishing, grande → exploding)
        Wh = np.random.randn(hidden_size, hidden_size) * init_std
        h  = np.random.randn(hidden_size) * 0.1
        
        # Gradient initial (delta en T)
        grad = np.ones(hidden_size) / np.sqrt(hidden_size)
        norms = [np.linalg.norm(grad)]
        
        for t in range(T):
            h = np.tanh(Wh @ h)
            dtanh = 1 - h**2          # dérivée de tanh
            J = np.diag(dtanh) @ Wh   # Jacobienne locale
            grad = J.T @ grad
            norms.append(np.linalg.norm(grad))
        
        norms_all.append(norms)
    return np.array(norms_all)

# Différentes initialisations
configs = [
    (0.05,  'Sous-critique (std=0.05)'),
    (0.125, 'Critique      (std=0.125)'),
    (0.3,   'Sur-critique  (std=0.3)'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Norme du gradient à travers le temps — RNN Vanilla', 
             fontsize=14, color='white', fontweight='bold', y=1.02)

for ax, (std, label) in zip(axes, configs):
    norms = simulate_gradient_norm(init_std=std, T=80)
    mean_n = norms.mean(axis=0)
    std_n  = norms.std(axis=0)
    t = np.arange(len(mean_n))
    
    ax.fill_between(t, mean_n - std_n, mean_n + std_n, alpha=0.2, color=COLORS['rnn'])
    ax.plot(t, mean_n, color=COLORS['rnn'], lw=2, label='Moyenne')
    ax.axhline(1.0, color=COLORS['gold'], lw=1, ls='--', alpha=0.7, label='Référence')
    ax.set_title(label, color='white', fontsize=10)
    ax.set_xlabel('Pas en arrière (T - t)')
    ax.set_ylabel('‖∂L/∂h_t‖')
    ax.set_yscale('log')
    ax.grid(True)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print('  On observe clairement le vanishing (std<0.125) ou l\'exploding (std>0.125)')

---
## 2. Le Constant Error Carousel (CEC) dans les LSTM

Hochreiter & Schmidhuber (1997) ont identifié le problème et proposé une solution élégante :

### Équations LSTM

| Gate | Formule |
|------|---------|
| **Forget** $f_t$ | $\sigma(W_f [h_{t-1}, x_t] + b_f)$ |
| **Input**  $i_t$ | $\sigma(W_i [h_{t-1}, x_t] + b_i)$ |
| **Candidat** $\tilde{c}_t$ | $\tanh(W_c [h_{t-1}, x_t] + b_c)$ |
| **Cell**   $c_t$ | $f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$ |
| **Output** $o_t$ | $\sigma(W_o [h_{t-1}, x_t] + b_o)$ |
| **Hidden** $h_t$ | $o_t \odot \tanh(c_t)$ |

### Le CEC : l'autoroute du gradient

La **clé** est la mise à jour de la cellule $c_t$ :

$$\frac{\partial c_t}{\partial c_{t-1}} = f_t$$

Le gradient de la loss par rapport à $c_t$ se propage ainsi :

$$\frac{\partial \mathcal{L}}{\partial c_t} = \frac{\partial \mathcal{L}}{\partial c_T} \cdot \prod_{k=t}^{T-1} f_{k+1}$$

Si $f_t \approx 1$ (forget gate ouverte), le gradient **ne décroît pas** → *Constant Error Carousel* !  
Le réseau **apprend** à ouvrir/fermer les gates selon le contexte.

In [ ]:
# ============================================================
# Visualisation schématique du flux de gradient
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

def draw_rnn_flow(ax):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 6)
    ax.set_title('RNN Vanilla — Flux du gradient', color='white', fontsize=12, fontweight='bold')
    ax.axis('off')
    
    T = 5
    xs = np.linspace(1.5, 8.5, T)
    y_h = 3.5
    
    # Coefficients de multiplication (simulés, décroissants)
    alpha = 0.45
    
    for i, x in enumerate(xs):
        # Nœud h_t
        circle = plt.Circle((x, y_h), 0.4, color=COLORS['rnn'], alpha=0.85, zorder=3)
        ax.add_patch(circle)
        ax.text(x, y_h, f'$h_{i+1}$', ha='center', va='center', fontsize=9, color='white', zorder=4)
        
        # Entrée x_t
        ax.annotate('', xy=(x, y_h-0.45), xytext=(x, y_h-1.3),
                    arrowprops=dict(arrowstyle='->', color='#888899', lw=1.2))
        ax.text(x, y_h-1.55, f'$x_{i+1}$', ha='center', va='center', fontsize=8, color='#888899')
        
        # Connexions horizontales + annotation du coefficient
        if i < T - 1:
            x2 = xs[i+1]
            factor = alpha ** (T - 1 - i)
            ax.annotate('', xy=(x2-0.42, y_h), xytext=(x+0.42, y_h),
                        arrowprops=dict(arrowstyle='->', color=COLORS['rnn'], lw=2))
            # Gradient en retour (en bas)
            y_g = 1.5
            ax.annotate('', xy=(x+0.42, y_g), xytext=(x2-0.42, y_g),
                        arrowprops=dict(arrowstyle='->', 
                                       color=plt.cm.RdYlGn(factor*2), lw=2.5))
            ax.text((x+x2)/2, y_g-0.35, f'×{factor:.3f}', ha='center', 
                    fontsize=8, color=plt.cm.RdYlGn(factor*2))
    
    ax.text(5, 0.6, '← Gradient backward (s\'atténue exponentiellement)', 
            ha='center', fontsize=9, color=COLORS['rnn'],
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#2a0a0a', edgecolor=COLORS['rnn']))
    
    ax.annotate('Gradient ≈ 0\n(information perdue)', xy=(xs[0], 1.5), 
                xytext=(xs[0]-0.3, 0.3),
                fontsize=8, color=COLORS['gold'],
                arrowprops=dict(arrowstyle='->', color=COLORS['gold'], lw=1))

def draw_lstm_flow(ax):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 6)
    ax.set_title('LSTM — Constant Error Carousel', color='white', fontsize=12, fontweight='bold')
    ax.axis('off')
    
    T = 5
    xs = np.linspace(1.5, 8.5, T)
    y_c = 4.3  # cell state
    y_h = 2.5  # hidden
    
    # Cell state highway (autoroute)
    ax.plot([xs[0]-0.5, xs[-1]+0.5], [y_c, y_c], 
            color=COLORS['lstm'], lw=3, alpha=0.3, ls='--', zorder=1)
    ax.text(5, y_c+0.4, 'Cell State $c_t$ — Autoroute du gradient (CEC)', 
            ha='center', fontsize=9, color=COLORS['lstm'], fontweight='bold')
    
    for i, x in enumerate(xs):
        # Cell state
        rect = mpatches.FancyBboxPatch((x-0.4, y_c-0.25), 0.8, 0.5,
                                        boxstyle='round,pad=0.05',
                                        facecolor=COLORS['lstm'], alpha=0.8, zorder=3)
        ax.add_patch(rect)
        ax.text(x, y_c, f'$c_{i+1}$', ha='center', va='center', fontsize=9, 
                color='white', zorder=4, fontweight='bold')
        
        # Hidden state
        circle = plt.Circle((x, y_h), 0.35, color=COLORS['purple'], alpha=0.85, zorder=3)
        ax.add_patch(circle)
        ax.text(x, y_h, f'$h_{i+1}$', ha='center', va='center', fontsize=8, 
                color='white', zorder=4)
        
        # Connexion c→h
        ax.annotate('', xy=(x, y_h+0.38), xytext=(x, y_c-0.28),
                    arrowprops=dict(arrowstyle='->', color='#888899', lw=1))
        
        if i < T - 1:
            x2 = xs[i+1]
            # Cell state forward
            ax.annotate('', xy=(x2-0.42, y_c), xytext=(x+0.42, y_c),
                        arrowprops=dict(arrowstyle='->', color=COLORS['lstm'], lw=2))
            # Forget gate annotation
            ax.text((x+x2)/2, y_c+0.15, '$f_t \\approx 1$', ha='center', 
                    fontsize=7.5, color=COLORS['gold'])
            
            # Gradient backward (stable !)
            y_g = 1.0
            ax.annotate('', xy=(x+0.42, y_g), xytext=(x2-0.42, y_g),
                        arrowprops=dict(arrowstyle='->', color=COLORS['green'], lw=2.5))
            ax.text((x+x2)/2, y_g-0.35, '×f_t≈1', ha='center', 
                    fontsize=8, color=COLORS['green'])
    
    ax.text(5, 0.15, '← Gradient stable grâce au CEC (pas de vanishing!)', 
            ha='center', fontsize=9, color=COLORS['green'],
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#0a2a0a', edgecolor=COLORS['green']))

draw_rnn_flow(axes[0])
draw_lstm_flow(axes[1])
plt.tight_layout()
plt.show()

---
## 3. Implémentation PyTorch avec hooks de gradient

In [ ]:
# ============================================================
# Modèles PyTorch avec enregistrement des gradients
# ============================================================
# num_layers=3 : gradients doivent traverser profondeur ET temps
# → vanishing encore plus sévère pour le RNN, contrasté par le LSTM

class VanillaRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=3):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        # Pile de RNNCells — unrolling manuel pour que chaque h_t soit
        # un nœud explicite du graphe et que les hooks se déclenchent
        self.rnn_cells = nn.ModuleList([
            nn.RNNCell(input_size if l == 0 else hidden_size, hidden_size)
            for l in range(num_layers)
        ])
        self.fc = nn.Linear(hidden_size, output_size)
        self.gradient_norms = []

    def forward(self, x):
        B, T, _ = x.shape
        hs = [torch.zeros(B, self.hidden_size, device=x.device)
              for _ in range(self.num_layers)]

        # On accroche les hooks sur la couche BASSE (couche 0) :
        # c'est elle qui souffre le plus du vanishing (profondeur + temps)
        bottom_hidden_states = []

        for t in range(T):
            inp = x[:, t, :]
            new_hs = []
            for l, cell in enumerate(self.rnn_cells):
                h_new = cell(inp, hs[l])
                inp   = h_new        # sortie couche l → entrée couche l+1
                new_hs.append(h_new)
            hs = new_hs
            bottom_hidden_states.append(hs[0])   # couche 0, pas t

        if self.training:
            self.gradient_norms = []
            for t_idx, h_t in enumerate(bottom_hidden_states):
                def make_hook(t_val):
                    def hook(grad):
                        self.gradient_norms.append((t_val, grad.norm().item()))
                    return hook
                h_t.register_hook(make_hook(t_idx))

        return self.fc(hs[-1])   # prédiction depuis la couche haute


class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=3):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        # Pile de LSTMCells — le CEC de chaque couche limite le vanishing
        self.lstm_cells = nn.ModuleList([
            nn.LSTMCell(input_size if l == 0 else hidden_size, hidden_size)
            for l in range(num_layers)
        ])
        self.fc = nn.Linear(hidden_size, output_size)
        self.gradient_norms = []

    def forward(self, x):
        B, T, _ = x.shape
        hs = [torch.zeros(B, self.hidden_size, device=x.device)
              for _ in range(self.num_layers)]
        cs = [torch.zeros(B, self.hidden_size, device=x.device)
              for _ in range(self.num_layers)]

        bottom_hidden_states = []

        for t in range(T):
            inp = x[:, t, :]
            new_hs, new_cs = [], []
            for l, cell in enumerate(self.lstm_cells):
                h_new, c_new = cell(inp, (hs[l], cs[l]))
                inp = h_new
                new_hs.append(h_new)
                new_cs.append(c_new)
            hs, cs = new_hs, new_cs
            bottom_hidden_states.append(hs[0])   # couche 0, pas t

        if self.training:
            self.gradient_norms = []
            for t_idx, h_t in enumerate(bottom_hidden_states):
                def make_hook(t_val):
                    def hook(grad):
                        self.gradient_norms.append((t_val, grad.norm().item()))
                    return hook
                h_t.register_hook(make_hook(t_idx))

        return self.fc(hs[-1])

print(' Modèles définis (3 couches empilées)')
print(f'   VanillaRNN (3L) : {sum(p.numel() for p in VanillaRNN(1,64,2).parameters()):,} paramètres')
print(f'   LSTMModel  (3L) : {sum(p.numel() for p in LSTMModel(1,64,2).parameters()):,} paramètres')


---
## 4. Tâche de mémorisation à long terme

Nous utilisons une tâche classique : **détecter un signal placé en début de séquence** après un long délai.  
Cela force le réseau à maintenir de l'information sur `T` pas de temps.

In [ ]:
# ============================================================
# Génération de données : tâche de mémoire à long terme
# ============================================================

def generate_memory_task(n_samples=2000, seq_len=50, signal_pos=3, noise_std=0.1):
    """
    Tâche : un signal binaire (±1) est placé à la position `signal_pos`.
    Le réseau doit prédire le signe de ce signal après `seq_len` pas.
    Tout le reste est du bruit gaussien.
    """
    X = np.random.randn(n_samples, seq_len, 1) * noise_std
    y = np.random.choice([-1, 1], size=n_samples)
    X[:, signal_pos, 0] = y  # signal à mémoriser
    labels = (y > 0).astype(np.int64)
    return torch.FloatTensor(X), torch.LongTensor(labels)

# Datasets pour différentes longueurs
SEQ_LENGTHS = [10, 30, 50, 80, 120]
datasets = {T: generate_memory_task(seq_len=T) for T in SEQ_LENGTHS}

# Visualisation d'un exemple
X_ex, y_ex = datasets[50]
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(X_ex[0, :, 0].numpy(), color=COLORS['blue'], lw=1.5, label='Séquence (bruit + signal)')
ax.axvline(3, color=COLORS['gold'], lw=2, ls='--', label=f'Signal à t=3 (classe={y_ex[0].item()})')
ax.axvline(49, color=COLORS['rnn'], lw=2, ls='--', label='Prédiction à t=49')
ax.fill_between(range(3, 50), -2, 2, alpha=0.05, color=COLORS['purple'])
ax.text(26, 1.2, '← délai à mémoriser (47 pas) →', ha='center', fontsize=10, color=COLORS['purple'])
ax.set_title('Exemple de tâche de mémorisation', fontsize=12, color='white')
ax.legend(fontsize=9)
ax.set_xlabel('Temps')
ax.set_ylabel('Valeur')
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Entraînement et collecte des normes de gradient
# ============================================================

def train_and_collect_gradients(model, X, y, n_epochs=30, lr=1e-3, batch_size=32):
    """Entraîne le modèle et collecte les normes de gradient par position temporelle."""
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)
    
    history = {'loss': [], 'acc': [], 'grad_norms_by_epoch': []}
    
    for epoch in range(n_epochs):
        model.train()
        epoch_loss, epoch_acc = 0, 0
        epoch_grads = {}
        
        for xb, yb in loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            epoch_acc  += (pred.argmax(1) == yb).float().mean().item()
            
            # Aggregation des gradients par position
            for t, norm in model.gradient_norms:
                if t not in epoch_grads:
                    epoch_grads[t] = []
                epoch_grads[t].append(norm)
        
        history['loss'].append(epoch_loss / len(loader))
        history['acc'].append(epoch_acc  / len(loader))
        history['grad_norms_by_epoch'].append(
            {t: np.mean(v) for t, v in epoch_grads.items()}
        )
    
    return history


# Entraînement sur séquence de longueur 50
print('Entraînement sur T=50...')
H = 64
X50, y50 = datasets[50]

rnn_model  = VanillaRNN(1, H, 2)
lstm_model = LSTMModel(1, H, 2)

hist_rnn  = train_and_collect_gradients(rnn_model,  X50, y50, n_epochs=40)
hist_lstm = train_and_collect_gradients(lstm_model, X50, y50, n_epochs=40)

print(f'RNN  — Accuracy finale : {hist_rnn["acc"][-1]:.3f}')
print(f'LSTM — Accuracy finale : {hist_lstm["acc"][-1]:.3f}')

In [ ]:
# ============================================================
# Visualisation principale : courbes d'apprentissage + gradient
# ============================================================

fig = plt.figure(figsize=(16, 12), facecolor='#0f0f1a')
gs  = GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

fig.suptitle('Vanishing Gradient : RNN Vanilla vs LSTM (T=50)', 
             fontsize=15, color='white', fontweight='bold')

# --- Courbe de loss ---
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(hist_rnn['loss'],  color=COLORS['rnn'],  lw=2, label='RNN')
ax1.plot(hist_lstm['loss'], color=COLORS['lstm'], lw=2, label='LSTM')
ax1.set_title('Courbe de Loss', color='white')
ax1.set_xlabel('Époque'); ax1.set_ylabel('CrossEntropy')
ax1.legend(); ax1.grid(True)

# --- Courbe d'accuracy ---
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(hist_rnn['acc'],  color=COLORS['rnn'],  lw=2, label='RNN')
ax2.plot(hist_lstm['acc'], color=COLORS['lstm'], lw=2, label='LSTM')
ax2.axhline(0.5, color=COLORS['gold'], lw=1, ls='--', alpha=0.7, label='Chance')
ax2.set_title('Accuracy (tâche mémoire T=50)', color='white')
ax2.set_xlabel('Époque'); ax2.set_ylabel('Accuracy')
ax2.set_ylim(0.3, 1.05); ax2.legend(); ax2.grid(True)

# --- Normes de gradient par position (dernière époque) ---
def extract_grad_profile(history):
    last_epoch = history['grad_norms_by_epoch'][-1]
    T_vals  = sorted(last_epoch.keys())
    norms   = [last_epoch[t] for t in T_vals]
    return T_vals, norms

ax3 = fig.add_subplot(gs[1, :])
t_rnn,  n_rnn  = extract_grad_profile(hist_rnn)
t_lstm, n_lstm = extract_grad_profile(hist_lstm)

ax3.semilogy(t_rnn,  n_rnn,  color=COLORS['rnn'],  lw=2.5, marker='o', ms=4, label='RNN  ‖∂L/∂h_t‖')
ax3.semilogy(t_lstm, n_lstm, color=COLORS['lstm'], lw=2.5, marker='s', ms=4, label='LSTM ‖∂L/∂h_t‖')
ax3.axvline(3, color=COLORS['gold'], lw=2, ls='--', alpha=0.8, label='Position du signal (t=3)')
ax3.fill_betweenx([min(n_rnn+n_lstm)*0.1, max(n_rnn+n_lstm)*10], 0, 3, 
                   alpha=0.1, color=COLORS['gold'])
ax3.set_title('Norme du gradient par position temporelle (après 40 époques)', 
              color='white', fontsize=12)
ax3.set_xlabel('Position temporelle t')
ax3.set_ylabel('‖∂L/∂h_t‖ (log scale)')
ax3.legend(fontsize=10); ax3.grid(True)
ax3.text(1.5, max(n_lstm)*0.5, 'Le gradient\ndoit atteindre ici', 
         ha='center', fontsize=9, color=COLORS['gold'],
         bbox=dict(boxstyle='round', facecolor='#2a2a00', edgecolor=COLORS['gold']))

# --- Ratio gradient LSTM/RNN ---
ax4 = fig.add_subplot(gs[2, :])
# Interpolation pour aligner les positions
t_common = sorted(set(t_rnn) & set(t_lstm))
rnn_dict  = dict(zip(t_rnn,  n_rnn))
lstm_dict = dict(zip(t_lstm, n_lstm))
ratios = [lstm_dict[t] / (rnn_dict[t] + 1e-30) for t in t_common]

colors_bar = [COLORS['lstm'] if r > 1 else COLORS['rnn'] for r in ratios]
ax4.bar(t_common, np.log10(np.array(ratios)+1e-10), color=colors_bar, alpha=0.8, width=0.6)
ax4.axhline(0, color='white', lw=1)
ax4.set_title('log₁₀(‖grad LSTM‖ / ‖grad RNN‖) — plus c\'est haut, plus LSTM conserve le gradient', 
              color='white')
ax4.set_xlabel('Position temporelle t')
ax4.set_ylabel('log₁₀(ratio)')
ax4.grid(True)

legend_elements = [
    mpatches.Patch(facecolor=COLORS['lstm'], label='LSTM > RNN (LSTM meilleur)'),
    mpatches.Patch(facecolor=COLORS['rnn'],  label='RNN > LSTM')
]
ax4.legend(handles=legend_elements, fontsize=9)

plt.show()

### Commentaire du graphe

**Courbes d'apprentissage (ligne du haut)**

Le RNN en rouge converge rapidement vers 100% d'accuracy autour de l'époque 5, mais **s'effondre ensuite complètement** pour revenir au niveau du hasard (50%). C'est un *faux départ* : le modèle a mémorisé un pattern superficiel dans les couches hautes, mais il n'a jamais appris à propager l'information depuis le début de la séquence. Le LSTM, lui, met plus de temps (~20 époques) mais **converge proprement et durablement**.

---

**Normes de gradient par position temporelle (panneau central)**

Les hooks sont posés sur la **couche 0** (la couche basse) — la plus difficile à atteindre, car le gradient doit traverser à la fois le temps (50 pas de BPTT) et la profondeur (3 couches). Le contraste est saisissant :

- **RNN (rouge)** : les normes sont de l'ordre de $10^{-19}$ aux positions précoces. C'est littéralement zéro numérique. Le gradient n'atteint **jamais** t=3, là où le signal à mémoriser est placé.
- **LSTM (vert)** : les normes restent stables autour de $10^{-6}$–$10^{-4}$ sur **l'ensemble de la séquence**. La forget gate ($f_t \approx 1$) maintient une autoroute de gradient à travers le temps.

Avec 3 couches, la dégradation du RNN devient $\approx \lambda_\text{temps}^{50} \times \lambda_\text{profondeur}^3 \approx 10^{-19}$ — bien pire qu'un modèle à une seule couche.

---

**Ratio LSTM / RNN (panneau bas)**

Aux positions t=0 à t=15, le LSTM transmet entre **10 et 13 ordres de grandeur** de gradient de plus que le RNN. À l'inverse, sur les derniers pas de temps (proches de la loss), le RNN récupère légèrement — mais ces positions ne contiennent que du bruit dans cette tâche, ce qui ne lui est d'aucune utilité.

> **Conclusion** : le RNN ne peut tout simplement pas apprendre à mémoriser un signal situé 47 pas avant la prédiction. Le LSTM le peut, grâce au Constant Error Carousel qui garantit un flux de gradient stable à travers le temps et la profondeur.


---
## 5. Expérience : accuracy en fonction de la longueur de séquence

In [ ]:
# ============================================================
# Benchmark : accuracy vs longueur de séquence
# ============================================================

results = {'rnn': [], 'lstm': []}

for T in SEQ_LENGTHS:
    print(f'T={T:3d}  ', end='')
    X, y = datasets[T]
    
    rnn_m  = VanillaRNN(1, 64, 2)
    lstm_m = LSTMModel(1, 64, 2)
    
    h_rnn  = train_and_collect_gradients(rnn_m,  X, y, n_epochs=60, lr=5e-3)
    h_lstm = train_and_collect_gradients(lstm_m, X, y, n_epochs=60, lr=5e-3)
    
    acc_rnn  = max(h_rnn['acc'][-10:])   # meilleure acc sur les 10 dernières époques
    acc_lstm = max(h_lstm['acc'][-10:])
    
    results['rnn'].append(acc_rnn)
    results['lstm'].append(acc_lstm)
    print(f'RNN={acc_rnn:.3f}  LSTM={acc_lstm:.3f}')

# Visualisation
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(SEQ_LENGTHS, results['rnn'],  'o-', color=COLORS['rnn'],  lw=2.5, ms=8, label='RNN Vanilla', zorder=3)
ax.plot(SEQ_LENGTHS, results['lstm'], 's-', color=COLORS['lstm'], lw=2.5, ms=8, label='LSTM',        zorder=3)
ax.fill_between(SEQ_LENGTHS, results['rnn'], results['lstm'], 
                alpha=0.15, color=COLORS['lstm'], label='Avantage LSTM')
ax.axhline(0.5, color=COLORS['gold'], lw=1.5, ls='--', alpha=0.7, label='Chance (50%)')
ax.axhline(0.9, color='white',        lw=1,   ls=':',  alpha=0.4, label='Seuil 90%')

ax.set_xlabel('Longueur de séquence T', fontsize=12)
ax.set_ylabel('Accuracy de classification', fontsize=12)
ax.set_title('Impact du vanishing gradient sur la mémoire à long terme', 
             fontsize=13, color='white', fontweight='bold')
ax.set_ylim(0.4, 1.05)
ax.legend(fontsize=10); ax.grid(True)

# Annotations
for T, r_rnn, r_lstm in zip(SEQ_LENGTHS, results['rnn'], results['lstm']):
    ax.annotate(f'{r_rnn:.2f}', (T, r_rnn),  textcoords='offset points', 
                xytext=(0, -18), ha='center', fontsize=8, color=COLORS['rnn'])
    ax.annotate(f'{r_lstm:.2f}', (T, r_lstm), textcoords='offset points', 
                xytext=(0, 8),   ha='center', fontsize=8, color=COLORS['lstm'])

plt.tight_layout()
plt.show()

---
## 6. Analyse des Gates LSTM : visualiser le CEC en action

In [ ]:
# ============================================================
# Visualisation des activations de gate sur un exemple
# ============================================================

class LSTMWithGateViz(nn.Module):
    """LSTM qui expose les activations de ses gates."""
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        # Gates manuelles pour accès aux activations
        self.Wi = nn.Linear(input_size + hidden_size, hidden_size)  # input gate
        self.Wf = nn.Linear(input_size + hidden_size, hidden_size)  # forget gate
        self.Wg = nn.Linear(input_size + hidden_size, hidden_size)  # cell gate
        self.Wo = nn.Linear(input_size + hidden_size, hidden_size)  # output gate
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x, return_gates=False):
        B, T, _ = x.shape
        h = torch.zeros(B, self.hidden_size)
        c = torch.zeros(B, self.hidden_size)
        
        f_gates, i_gates, o_gates = [], [], []
        
        for t in range(T):
            combined = torch.cat([x[:, t, :], h], dim=1)
            f = torch.sigmoid(self.Wf(combined))
            i = torch.sigmoid(self.Wi(combined))
            g = torch.tanh(self.Wg(combined))
            o = torch.sigmoid(self.Wo(combined))
            c = f * c + i * g
            h = o * torch.tanh(c)
            
            f_gates.append(f.detach())
            i_gates.append(i.detach())
            o_gates.append(o.detach())
        
        out = self.fc(h)
        if return_gates:
            return out, torch.stack(f_gates, 1), torch.stack(i_gates, 1), torch.stack(o_gates, 1)
        return out


# Entraîner ce modèle
viz_model = LSTMWithGateViz(1, 32, 2)
optimizer = optim.Adam(viz_model.parameters(), lr=5e-3)
criterion = nn.CrossEntropyLoss()
X_tr, y_tr = datasets[50]
loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)

for epoch in range(80):
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(viz_model(xb), yb)
        loss.backward()
        optimizer.step()

print(f'Modèle entraîné — évaluation...')
viz_model.eval()
with torch.no_grad():
    preds, f_g, i_g, o_g = viz_model(X_tr[:100], return_gates=True)
    acc = (preds.argmax(1) == y_tr[:100]).float().mean().item()
print(f'Accuracy : {acc:.3f}')

# Plot des gates moyennées
f_mean = f_g.mean(dim=(0, 2)).numpy()  # moyenne sur batch et neurones
i_mean = i_g.mean(dim=(0, 2)).numpy()
o_mean = o_g.mean(dim=(0, 2)).numpy()

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
fig.suptitle('Activations des gates LSTM sur la tâche de mémorisation (T=50)', 
             fontsize=13, color='white', fontweight='bold')

gate_data = [
    (f_mean, 'Forget Gate $f_t$',  COLORS['rnn'],  'Proche de 1 = mémorise, proche de 0 = oublie'),
    (i_mean, 'Input Gate $i_t$',   COLORS['green'], 'S\'active quand nouvelle info à stocker'),
    (o_mean, 'Output Gate $o_t$',  COLORS['purple'],'Contrôle ce qui est lu depuis $c_t$'),
]

for ax, (data, title, color, desc) in zip(axes, gate_data):
    ax.plot(data, color=color, lw=2.5)
    ax.fill_between(range(len(data)), 0, data, alpha=0.2, color=color)
    ax.axvline(3, color=COLORS['gold'], lw=2, ls='--', alpha=0.9)
    ax.axhline(0.5, color='white', lw=1, ls=':', alpha=0.4)
    ax.set_ylabel(title, fontsize=11)
    ax.set_ylim(0, 1)
    ax.grid(True)
    ax.text(45, 0.85, desc, ha='right', fontsize=9, color=color,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='#1a1a2e', edgecolor=color, alpha=0.8))
    ax.text(3.2, 0.1, 'Signal', fontsize=8, color=COLORS['gold'])

axes[-1].set_xlabel('Position temporelle t', fontsize=11)
plt.tight_layout()
plt.show()

---
## 7. Analyse de la Jacobienne : valeurs singulières

In [ ]:
# ============================================================
# Spectre des valeurs singulières des matrices de transition
# ============================================================

def get_jacobian_singular_values(model, x_seq):
    """Calcule les valeurs singulières de dh_t/dh_{t-1} pour chaque couche."""
    model.eval()
    # On analyse la couche 0 (bas) — la plus affectée par le vanishing
    if isinstance(model, VanillaRNN):
        Wh = model.rnn_cells[0].weight_hh.detach().numpy()
        Wx = model.rnn_cells[0].weight_ih.detach().numpy()
        b  = (model.rnn_cells[0].bias_hh + model.rnn_cells[0].bias_ih).detach().numpy()

        h = np.zeros(model.hidden_size)
        sv_history = []
        for t in range(x_seq.shape[0]):
            x = x_seq[t]
            z  = Wh @ h + Wx @ x + b
            h  = np.tanh(z)
            dtanh = 1 - h**2
            J = np.diag(dtanh) @ Wh
            sv = np.linalg.svd(J, compute_uv=False)
            sv_history.append(sv)

        return np.array(sv_history)   # (T, hidden_size)

    return None

# Extraire une séquence exemple
x_sample    = X50[0, :, 0].numpy()
x_sample_2d = x_sample[:, np.newaxis]  # (T, 1)

sv_rnn = get_jacobian_singular_values(rnn_model, x_sample_2d)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Heatmap des valeurs singulières
im = axes[0].imshow(sv_rnn[:, :20].T, aspect='auto', cmap='RdYlGn',
                     vmin=0, vmax=1.5, origin='lower')
axes[0].set_title('Valeurs singulières de J_t = ∂h_t/∂h_{t-1}\n(RNN couche 0 — 20 premières)', 
                   color='white')
axes[0].set_xlabel('Pas de temps t')
axes[0].set_ylabel('Rang de la valeur singulière')
axes[0].axvline(3, color=COLORS['gold'], lw=2, ls='--')
plt.colorbar(im, ax=axes[0], label='Valeur singulière')

# Produit cumulé de la valeur singulière max
sv_max      = sv_rnn[:, 0]
cum_product = np.cumprod(sv_max[::-1])[::-1]

axes[1].semilogy(cum_product, color=COLORS['rnn'], lw=2.5, label='∏ σ_max couche 0 (RNN)')
axes[1].axhline(1.0, color=COLORS['gold'], lw=1, ls='--', label='σ=1 (stable)')
axes[1].axhline(1e-3, color='white', lw=1, ls=':', alpha=0.5, label='Seuil vanishing')
axes[1].fill_between(range(len(cum_product)), 1e-10, 1e-3, alpha=0.1, color=COLORS['rnn'])
axes[1].text(25, 1e-5, 'Zone vanishing gradient\n(3 couches × T pas)', ha='center', 
             fontsize=9, color=COLORS['rnn'],
             bbox=dict(boxstyle='round', facecolor='#2a0a0a', edgecolor=COLORS['rnn']))
axes[1].set_title('Produit cumulé σ_max — couche 0 (bas)', color='white')
axes[1].set_xlabel('Position temporelle t')
axes[1].set_ylabel('Amplification du gradient (log)')
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()


---
## 8. Résumé : Tableau comparatif

In [ ]:
# ============================================================
# Tableau récapitulatif visuel
# ============================================================

fig, ax = plt.subplots(figsize=(14, 6), facecolor='#0f0f1a')
ax.axis('off')
ax.set_facecolor('#0f0f1a')

headers = ['Aspect', 'RNN Vanilla', 'LSTM']
rows = [
    ['État récurrent',       'h_t = tanh(W_h h_{t-1} + ...)',      'c_t = f_t⊙c_{t-1} + i_t⊙g_t'],
    ['Jacobienne ∂h_t/∂h_{t-1}', 'diag(1-h²)·W_h (produit de matrices)', 'f_t (produit de scalaires)'],
    ['Flux du gradient',     'Exponentiel : α^T → 0',               'Additif via CEC : contrôlé'],
    ['Mémoire à long terme', ' Dépend de l\'init et T',             ' Mémorise indépendamment de T'],
    ['Gates',                '—',                                    'Forget / Input / Output'],
    ['Paramètres (H=64)',    f'{sum(p.numel() for p in rnn_model.parameters()):,}', 
                              f'{sum(p.numel() for p in lstm_model.parameters()):,}'],
    ['Acc T=50 (expérience)', f'{hist_rnn["acc"][-1]:.1%}',          f'{hist_lstm["acc"][-1]:.1%}'],
]

all_data = [headers] + rows
n_rows = len(all_data)
n_cols = 3

col_widths = [0.28, 0.36, 0.36]
col_starts = [0.02, 0.30, 0.66]
row_height = 0.12

for r, row in enumerate(all_data):
    y = 1.0 - r * row_height - 0.05
    is_header = (r == 0)
    
    for c, (cell, x, w) in enumerate(zip(row, col_starts, col_widths)):
        if is_header:
            bg = '#16213e'
            fc = COLORS['gold']
            fs = 11
            fw = 'bold'
        elif c == 0:
            bg = '#1a1a2e'
            fc = '#aaaacc'
            fs = 9
            fw = 'bold'
        elif c == 1:
            bg = '#2a1515' if r % 2 else '#231010'
            fc = COLORS['rnn']
            fs = 8.5
            fw = 'normal'
        else:
            bg = '#152a15' if r % 2 else '#102310'
            fc = COLORS['lstm']
            fs = 8.5
            fw = 'normal'
        
        rect = mpatches.FancyBboxPatch((x, y-row_height+0.01), w-0.01, row_height-0.01,
                                        boxstyle='round,pad=0.005', 
                                        facecolor=bg, edgecolor='#334', lw=0.5,
                                        transform=ax.transAxes, clip_on=False)
        ax.add_patch(rect)
        ax.text(x + w/2, y - row_height/2, cell, 
                ha='center', va='center', fontsize=fs,
                color=fc, fontweight=fw,
                transform=ax.transAxes, clip_on=False,
                wrap=True)

ax.set_title('Comparaison RNN Vanilla vs LSTM : le problème et la solution', 
             fontsize=13, color='white', fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

---
## 9. Conclusion

### Ce que nous avons montré

| # | Observation |
|---|-------------|
| 1 | Le gradient d'un **RNN vanilla** décroît exponentiellement avec la distance temporelle : $\|\nabla_{h_t}\mathcal{L}\| \propto \lambda^{T-t}$ avec $\lambda < 1$ |
| 2 | La **Jacobienne** $\partial h_{t+1}/\partial h_t = \text{diag}(1 - h^2)\cdot W_h$ a des valeurs singulières $< 1$ pour des poids typiques |
| 3 | Le **LSTM** résout ce problème via le **Constant Error Carousel** : la mise à jour additive $c_t = f_t \odot c_{t-1} + \ldots$ permet au gradient de traverser $T$ pas sans s'atténuer |
| 4 | Empiriquement, le LSTM **surpasse massivement** le RNN vanilla sur des tâches nécessitant une mémoire longue (T > 20) |
| 5 | La **forget gate** est la clé : quand $f_t \approx 1$, le gradient se propage librement ; le réseau apprend quand ouvrir/fermer cette gate |

### Pour aller plus loin
- **GRU** (Cho et al., 2014) : version simplifiée du LSTM (2 gates au lieu de 3)
- **Attention / Transformers** : abandonnent complètement la récurrence pour des connexions directes
- **Highway Networks / ResNets** : même idée appliquée aux réseaux profonds (skip connections)
- **Gradient clipping** : mitigation empirique du gradient exploding dans les RNN